# HBNN: single-notebook version of `hbnn_training.py` (Kaggle edition)

Self-contained port of `hbnn_training.py` from
[TowardsUncertaintyAwareWorkloadPrediction](https://github.com/andreareds/TowardsUncertaintyAwareWorkloadPrediction)
(multibivariate, window 288, horizon 2). **HBNN**: CNN + LSTM + MLP, a Bayesian *variational dense* layer (full-covariance Gaussian posterior over its weights, N(0, I) prior) and a Gaussian output head (mean + std).

It inlines `models/HBNN.py`, `util/multipleDataset.py`, `util/save_results.py` and the
`hyperparams/multibivariate/HBNN-w288-h2.csv` values. **`tensorflow_probability` is not needed** (see notes at the bottom).
It also exports benchmark-style Gaussian service-level metrics (SR/TPR) and plots.

**Kaggle setup**
- Settings → **Accelerator → GPU T4 x2 or P100** (TensorFlow uses one GPU).
- Attach your dataset (**Input → Add Input**, e.g. `ML_PROJ_DATASET` with the 12 CSVs). No internet needed.
- Results go to `/kaggle/working/res` and `time/` (Output tab after *Save Version → Save & Run All*).
- **Resumable:** re-run after a timeout and finished runs are skipped.
- Runs one training iteration, capped at 500 epochs, with early stopping after 20 epochs without validation-loss improvement.

In [ ]:
import os, math
from datetime import datetime
from statistics import NormalDist

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, ops
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.utils import shuffle

# Work in /kaggle/working (the only persisted, writable directory on Kaggle)
if os.path.isdir('/kaggle/working'):
    os.chdir('/kaggle/working')
print('cwd:', os.getcwd())

assert int(keras.__version__.split('.')[0]) >= 3, 'This notebook needs Keras 3 (TF >= 2.16, standard on Kaggle).'
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print('TF', tf.__version__, '| Keras', keras.__version__, '| GPUs:', gpus)
if not gpus:
    print('WARNING: no GPU found. Enable one in Settings -> Accelerator, or training will be very slow.')

## 1. Config  (defaults = original script)

In [ ]:
MODEL = 'HBNN'
WIN = 288                  # input window (5-min steps -> 24h)
HORIZON = 2
CLUSTERS = ['gc19_a', 'gc19_b', 'gc19_c', 'gc19_d', 'gc19_e', 'gc19_f', 'gc19_g', 'gc19_h',
            'gc11', 'ali18', 'ali20_c', 'ali20_g']
ITERATIONS = 1             # one training run per train split
TRAIN_SPLITS = [0.8]       # original loops [0.8, 0.6, 0.4, 0.2]; see notes at the bottom
SUFFIX = "MULTIBI"

EPOCHS_OVERRIDE = None     # optionally override the 500-epoch maximum
RESTORE_BEST = False       # original does NOT restore best-val weights after early stopping
VERBOSE = 0                # keras progress bar off (it floods Kaggle logs); see LOG_EVERY
LOG_EVERY = 25             # print loss every N epochs
PRED_BATCH = 2048          # batch size used only for producing predictions

# Hyperparameters from hyperparams/multibivariate/HBNN-w288-h2.csv
P = dict(
    first_conv_dim=16, first_conv_kernel=3, first_conv_activation='relu', cnn_layers=1,
    second_lstm_dim=32, first_dense_dim=8, first_dense_activation='relu',
    mlp_units=[32, 32],
    dense_kernel_init='he_normal',   # present in the CSV but never used by the original model
    batch_size=128, epochs=500, patience=20,
    optimizer='adam', lr=1e-3, momentum=0.9, decay=1e-4,
)
if EPOCHS_OVERRIDE:
    P['epochs'] = EPOCHS_OVERRIDE

os.makedirs('res', exist_ok=True)
os.makedirs('time', exist_ok=True)

## 2. Data  (reads the CSVs from your attached Kaggle dataset, e.g. `ML_PROJ_DATASET`)

In [ ]:
# Finds <cluster>.csv anywhere under /kaggle/input, so the dataset's slug/folder name doesn't matter.
INPUT_ROOT = '/kaggle/input' if os.path.isdir('/kaggle/input') else '.'
found = {}
for root, _, files in os.walk(INPUT_ROOT):
    for fn in files:
        if fn.endswith('.csv') and fn[:-4] in CLUSTERS:
            found.setdefault(fn[:-4], os.path.join(root, fn))

missing = [c for c in CLUSTERS if c not in found]
if missing:
    raise FileNotFoundError(f'Not found under {INPUT_ROOT}: {missing}. '
                            'Add your dataset via Input -> Add Input, or trim CLUSTERS.')
DATA_PATH = found
print('data ready:', len(DATA_PATH), 'files, e.g.', next(iter(DATA_PATH.values())))

_chk = pd.read_csv(next(iter(DATA_PATH.values())), nrows=2)
assert {'avgcpu', 'avgmem'} <= set(_chk.columns), f'unexpected columns: {list(_chk.columns)}'

## 3. Dataset builder  (port of `util/multipleDataset.py`)
Per cluster: sliding windows (`X[i] = series[i:i+WIN]`, `y[i] = series[i+WIN+HORIZON]`), first `train_split`
fraction as train, then all clusters are concatenated and the train set is shuffled.

In [ ]:
def windowed(series, win, horizon):
    n = len(series) - win - horizon
    X = sliding_window_view(series, win, axis=0)[:n].transpose(0, 2, 1)   # (n, win, channels)
    y = series[win + horizon: win + horizon + n]                          # (n, channels)
    return X, y

def build_dataset(clusters, win, horizon, train_split):
    Xtr, ytr, Xte, yte = [], [], [], []
    for c in clusters:
        df = pd.read_csv(DATA_PATH[c])
        series = df[['avgcpu', 'avgmem']].to_numpy(dtype=np.float32)
        split = int(len(series) * train_split)       # same split rule as the original
        X, y = windowed(series, win, horizon)
        Xtr.append(X[:split]); ytr.append(y[:split])
        Xte.append(X[split:]); yte.append(y[split:])
    X_train, y_train = np.concatenate(Xtr), np.concatenate(ytr)
    X_test, y_test = np.concatenate(Xte), np.concatenate(yte)
    X_train, y_train = shuffle(X_train, y_train, random_state=0)
    print('Training', X_train.shape, 'Testing', X_test.shape)
    return X_train, y_train, X_test, y_test

## 4. Model  (port of `models/HBNN.py`, replacing the `tensorflow_probability` pieces)
Output is `[mean, std]`; the loss is the Gaussian negative log-likelihood, as in the original.

In [ ]:
LOG2PI = math.log(2.0 * math.pi)

def make_optimizer(p):
    o, lr = p['optimizer'], p['lr']
    if o == 'adam':
        # legacy Adam(decay=d) means lr_t = lr / (1 + d * step); InverseTimeDecay reproduces that
        # and works on Keras 3 (where `decay=` was removed).
        sched = keras.optimizers.schedules.InverseTimeDecay(lr, decay_steps=1, decay_rate=p['decay'])
        return keras.optimizers.Adam(learning_rate=sched)
    if o == 'rmsprop': return keras.optimizers.RMSprop(learning_rate=lr)
    if o == 'nadam':   return keras.optimizers.Nadam(learning_rate=lr)
    if o == 'sgd':     return keras.optimizers.SGD(learning_rate=lr, momentum=p['momentum'])
    raise ValueError(o)

# ---- Gaussian output head (replaces tfp.layers.IndependentNormal + negative_loglikelihood) ----
class NormalHead(layers.Layer):
    """params (B, 2d) -> [loc, softplus(raw_scale)] (B, 2d); same parameterisation as IndependentNormal."""
    def call(self, params):
        loc, raw = ops.split(params, 2, axis=-1)
        return ops.concatenate([loc, ops.softplus(raw)], axis=-1)

def gaussian_nll(y_true, y_pred):
    """-log N(y | loc, scale), summed over the output dims (= -Independent(Normal).log_prob(y))."""
    d = y_true.shape[-1]
    loc, scale = y_pred[:, :d], y_pred[:, d:]
    z = (y_true - loc) / scale
    return ops.sum(0.5 * LOG2PI + ops.log(scale) + 0.5 * ops.square(z), axis=-1)

def mse(y_true, y_pred):   # metrics are computed on the predicted mean
    d = y_true.shape[-1]
    return ops.mean(ops.square(y_true - y_pred[:, :d]), axis=-1)

def mae(y_true, y_pred):
    d = y_true.shape[-1]
    return ops.mean(ops.abs(y_true - y_pred[:, :d]), axis=-1)


class VariationalDense(layers.Layer):
    """Port of tfp.layers.DenseVariational as used by the original HBNN:
       posterior q(w) = full-covariance Gaussian MultivariateNormalTriL over [kernel, bias] (zero-init params,
       scale diag = softplus(.) + 1e-5), prior p(w) = N(0, I). ONE weight sample w ~ q is drawn per forward call and
       shared by the whole batch; the Monte-Carlo KL = kl_weight * (log q(w) - log p(w)) is added to the loss."""
    def __init__(self, units, kl_weight, activation='relu', **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.kl_weight = float(kl_weight)
        self.activation = keras.activations.get(activation)
        self.seed_gen = keras.random.SeedGenerator()

    def build(self, input_shape):
        self.d_in = int(input_shape[-1])
        n = self.n = self.d_in * self.units + self.units
        self.loc = self.add_weight(shape=(n,), initializer='zeros', name='loc')
        self.scale_params = self.add_weight(shape=(n, n), initializer='zeros', name='scale_tril')  # lower tri used
        self.cached_w = self.add_weight(shape=(n,), initializer='zeros', trainable=False, name='cached_w')
        self.use_cached = self.add_weight(shape=(), initializer='zeros', trainable=False, name='use_cached')
        self._mask = np.tril(np.ones((n, n), dtype='float32'), -1)
        self.built = True

    def _scale(self):
        W = ops.convert_to_tensor(self.scale_params)
        return W * self._mask, ops.softplus(ops.diagonal(W)) + 1e-5      # strictly-lower part, diagonal

    def _draw(self, eps=None):
        """Sample w = loc + L eps; return (w, MC estimate of KL(q || p) at that w)."""
        if eps is None:
            eps = keras.random.normal((self.n,), seed=self.seed_gen)
        lower, d = self._scale()
        w = ops.convert_to_tensor(self.loc) + ops.matmul(lower, eps[:, None])[:, 0] + d * eps
        kl = -0.5 * ops.sum(ops.square(eps)) - ops.sum(ops.log(d)) + 0.5 * ops.sum(ops.square(w))
        return w, kl

    def _apply(self, x, w):
        k = self.d_in * self.units
        kernel = ops.reshape(w[:k], (self.d_in, self.units))
        return self.activation(ops.matmul(x, kernel) + w[k:])

    def call(self, x):
        w_new, kl = self._draw()
        w = ops.where(ops.convert_to_tensor(self.use_cached) > 0.5, ops.convert_to_tensor(self.cached_w), w_new)
        self.add_loss(self.kl_weight * kl)
        return self._apply(x, w)

    # The original calls model(X) once on the whole set => a single weight sample for all of it.
    # We predict in chunks, so freeze one sample and reuse it for every chunk.
    def freeze_sample(self):
        w, _ = self._draw()
        self.cached_w.assign(w)
        self.use_cached.assign(1.0)

    def unfreeze_sample(self):
        self.use_cached.assign(0.0)

In [ ]:
def build_model(input_shape, out_dim, p, n_train):
    keras.backend.clear_session()
    inp = keras.Input(shape=input_shape)
    x = inp
    for _ in range(p['cnn_layers']):
        x = layers.Conv1D(filters=p['first_conv_dim'], kernel_size=p['first_conv_kernel'],
                          strides=1, padding='causal', activation=p['first_conv_activation'])(x)
    x = layers.LSTM(p['second_lstm_dim'])(x)
    for dim in p['mlp_units']:
        x = layers.Dense(dim, activation='relu')(x)
    vlayer = VariationalDense(p['first_dense_dim'], kl_weight=1.0 / n_train,      # same 1/N_train weight as the original
                              activation=p['first_dense_activation'], name='var')
    x = vlayer(x)
    x = layers.Dense(2 * out_dim)(x)
    out = NormalHead()(x)
    model = keras.Model(inp, out)
    model.compile(loss=gaussian_nll, optimizer=make_optimizer(p), metrics=[mse, mae], jit_compile=False)
    return model, vlayer

def predict_dist(model, vlayer, X):
    """Returns (mean, std), each (N, 2)."""
    vlayer.freeze_sample()        # one weight sample for the whole set, like the original model(X)
    try:
        out = model.predict(X, batch_size=PRED_BATCH, verbose=0)
    finally:
        vlayer.unfreeze_sample()
    d = out.shape[1] // 2
    return out[:, :d], out[:, d:]

class LogEvery(keras.callbacks.Callback):
    def __init__(self, n): super().__init__(); self.n = n
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % self.n == 0:
            print(f"  epoch {epoch + 1}: loss={logs['loss']:.5f}  val_loss={logs['val_loss']:.5f}", flush=True)

def train_one(X_train, y_train, p):
    model, vlayer = build_model(X_train.shape[1:], y_train.shape[1], p, n_train=X_train.shape[0])
    es = keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', verbose=1,
                                       patience=p['patience'], restore_best_weights=RESTORE_BEST)
    t0 = datetime.now()
    history = model.fit(X_train, y_train, epochs=p['epochs'], batch_size=p['batch_size'],
                        validation_split=0.2, verbose=VERBOSE, callbacks=[es, LogEvery(LOG_EVERY)])
    return model, vlayer, history, datetime.now() - t0

## 5. Result saving  (port of `util/save_results.py`)

In [ ]:
def save_uncertainty_csv(mean, std, labels, filename):
    pd.DataFrame({'avgcpu': mean[:, 0], 'stdavgcpu': std[:, 0], 'labelsavgcpu': labels[:, 0],
                  'avgmem': mean[:, 1], 'stdavgmem': std[:, 1], 'labelsavgmem': labels[:, 1]}
                 ).to_csv(f'res/output_{filename}.csv')

def save_errors(mses, maes, filename):
    pd.DataFrame({'MSE': mses, 'MAE': maes}).to_csv(f'res/errors_{filename}.csv')


SERVICE_LEVELS = [level / 100 for level in range(90, 100)] + [0.995]

def service_level_metrics(mean, std, actual):
    """Benchmark-style SR/TPR from marginal Gaussian upper quantiles."""
    rows = []
    for channel, resource in enumerate(('avgcpu', 'avgmem')):
        y = actual[:, channel]
        mu = mean[:, channel]
        sigma = np.maximum(std[:, channel], 0.0)
        demand = float(np.sum(y))
        for level in SERVICE_LEVELS:
            upper = mu + NormalDist().inv_cdf(level) * sigma
            tail = (1.0 - level) / 2.0
            lower_interval = mu + NormalDist().inv_cdf(tail) * sigma
            upper_interval = mu + NormalDist().inv_cdf(1.0 - tail) * sigma
            sr = float(np.mean(y <= upper) * 100.0)
            rows.append({'resource': resource, 'service_level': level, 'SR': sr,
                         'TPR': float(100.0 * np.sum(upper) / demand) if demand > 0 else float('nan'),
                         'coverage': float(np.mean((y >= lower_interval) & (y <= upper_interval)) * 100.0),
                         'calibration_error': sr - level * 100.0,
                         'mean_interval_width': float(np.mean(upper_interval - lower_interval))})
    return pd.DataFrame(rows)


## 6. Run  (resumable: finished runs are skipped on re-run)

In [ ]:
results = {}
for ts in TRAIN_SPLITS:
    name = f"{SUFFIX}{MODEL}-w{WIN}-h{HORIZON}-ts{int(ts * 100)}"
    print('=' * 20, name, '=' * 20)
    X_train, y_train, X_test, y_test = build_dataset(CLUSTERS, WIN, HORIZON, ts)
    print(P)

    mses, maes, training_times, inference_times, service_frames = [], [], [], [], []
    best_mse, best = float('inf'), None

    for it in range(ITERATIONS):
        test_csv  = f'res/output_{name}-run-{it}.csv'
        train_csv = f'res/output_train-{name}-run-{it}.csv'
        done_csv  = f'time/{name}-run-{it}-times.csv'          # written last = "run finished" marker

        resume = all(os.path.exists(f) for f in (test_csv, train_csv, done_csv))
        if resume:
            marker = pd.read_csv(done_csv)
            required = {'training_s', 'inference_s', 'epochs', 'patience'}
            resume = (len(marker) == 1 and required.issubset(marker.columns)
                      and marker.loc[0, 'epochs'] == P['epochs']
                      and marker.loc[0, 'patience'] == P['patience'])

        if resume:
            d = pd.read_csv(test_csv)
            mean = d[['avgcpu', 'avgmem']].to_numpy(dtype=np.float32)
            std = d[['stdavgcpu', 'stdavgmem']].to_numpy(dtype=np.float32)
            t = marker.iloc[0]
            tr_t, inf_t = pd.to_timedelta(t['training_s'], unit='s'), pd.to_timedelta(t['inference_s'], unit='s')
            print(f'run {it}: already done, loaded from disk')
        else:
            print('ITERATION:', it, 'HORIZON:', HORIZON, 'WIN:', WIN, flush=True)
            model, vlayer, history, tr_t = train_one(X_train, y_train, P)

            t1 = datetime.now()
            mean, std = predict_dist(model, vlayer, X_test)
            inf_t = (datetime.now() - t1) / len(X_test)

            tr_mean, tr_std = predict_dist(model, vlayer, X_train)
            save_uncertainty_csv(tr_mean, tr_std, y_train, f'train-{name}-run-{it}')
            save_uncertainty_csv(mean, std, y_test, f'{name}-run-{it}')
            pd.DataFrame({'training_s': [tr_t.total_seconds()],
                          'inference_s': [inf_t.total_seconds()],
                          'epochs': [P['epochs']], 'patience': [P['patience']]}).to_csv(done_csv, index=False)

        service_metrics = service_level_metrics(mean, std, y_test)
        service_metrics.to_csv(f'res/service_{name}-run-{it}.csv', index=False)
        service_frames.append(service_metrics)
        training_times.append(tr_t); inference_times.append(inf_t)
        mse_ = mean_squared_error(y_test, mean)
        mae_ = mean_absolute_error(y_test, mean)
        mses.append(mse_); maes.append(mae_)
        print(f'run {it}: test MSE={mse_:.6f}  MAE={mae_:.6f}  mean std={std.mean():.4f}  train time={tr_t}', flush=True)

        if mse_ < best_mse:
            best_mse, best = mse_, (mean, std)

    pd.DataFrame({'time': training_times}).to_csv(f'time/{name}training_time.csv')
    pd.DataFrame({'time': inference_times}).to_csv(f'time/{name}inference_time.csv')
    save_errors(mses, maes, name)
    service_metrics = pd.concat(service_frames, ignore_index=True)
    service_metrics.to_csv(f'res/service_{name}.csv', index=False)
    service_summary = (service_metrics.groupby(['resource', 'service_level'], as_index=False)
                       .agg(SR=('SR', 'mean'), TPR=('TPR', 'mean')))
    results[ts] = dict(mses=mses, maes=maes, y_test=y_test, mean=best[0], std=best[1], service_metrics=service_summary)
    print(f'{name}: MSE {np.mean(mses):.6f} +/- {np.std(mses):.6f} | MAE {np.mean(maes):.6f} +/- {np.std(maes):.6f}')

## 7. Quick look at the best run (mean ± 2σ)

In [ ]:
import matplotlib.pyplot as plt
r = results[TRAIN_SPLITS[0]]
n = min(500, len(r['y_test']))
x = np.arange(n)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
for ax, i, title in zip(axes, (0, 1), ('avg CPU', 'avg memory')):
    m, s = r['mean'][-n:, i], r['std'][-n:, i]
    ax.plot(x, r['y_test'][-n:, i], label='actual')
    ax.plot(x, m, label=f'{MODEL} mean')
    ax.fill_between(x, m - 2 * s, m + 2 * s, alpha=0.25, label='±2σ')
    ax.set_title(title + f' (last {n} test points)'); ax.legend()
plt.tight_layout(); plt.show()

## 8. Service-level metrics (SR / TPR)

The CSV `res/service_<run-name>.csv` contains SR and TPR for service levels 90% through 99% and 99.5%, using Gaussian upper quantiles from the model's predicted mean and standard deviation. The metrics are computed over the pooled test set; they are not per-trace values. Two plots are saved under `res/plots/`.

In [ ]:
os.makedirs('res/plots', exist_ok=True)
for ts in TRAIN_SPLITS:
    service = results[ts]['service_metrics']
    suffix = f'ts{int(ts * 100)}'

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, resource in zip(axes, ('avgcpu', 'avgmem')):
        data = service[service['resource'] == resource].sort_values('service_level')
        ax.plot(data['service_level'] * 100, data['SR'], marker='o', label='HBNN')
        ax.plot([90, 99.5], [90, 99.5], 'k--', label='Ideal calibration')
        ax.set(title=f'{resource}: confidence vs. SR', xlabel='Target service level (%)')
        ax.grid(alpha=0.25); ax.legend()
    axes[0].set_ylabel('Success rate, SR (%)')
    fig.tight_layout()
    fig.savefig(f'res/plots/service_confidence_vs_sr_{suffix}.png', dpi=150)
    plt.show(); plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for ax, resource in zip(axes, ('avgcpu', 'avgmem')):
        data = service[service['resource'] == resource].sort_values('TPR')
        ax.plot(data['TPR'], data['SR'], marker='o', label='HBNN')
        ax.set(title=f'{resource}: TPR vs. SR', xlabel='Predicted resources / actual demand, TPR (%)')
        ax.grid(alpha=0.25); ax.legend()
    axes[0].set_ylabel('Success rate, SR (%)')
    fig.tight_layout()
    fig.savefig(f'res/plots/service_tpr_vs_sr_{suffix}.png', dpi=150)
    plt.show(); plt.close(fig)

## Differences from the original script
- **Same:** data windowing and per-cluster split, shuffle (`random_state=0`), architecture, hyperparameters, NLL loss / optimizer,
  early stopping on `val_loss` (no best-weight restore), one run, and the `res/` + `time/` outputs (mean, std and labels per resource).
- **`tensorflow_probability` replaced.** `tfp.layers` doesn't work with Keras 3 (Kaggle's default), so the pieces are re-implemented
  in plain Keras and were checked numerically against TFP: the Gaussian NLL loss (`IndependentNormal`, std = softplus), and `DenseVariational`
  (full-covariance Gaussian posterior with zero-initialised params, N(0, I) prior, one weight sample per forward pass shared by the batch,
  Monte-Carlo KL weighted by `1/N_train`). Max differences vs TFP were ~1e-6 (float32 round-off).
- **Predictions use one weight sample** for the whole test set (and a fresh one for the train set), exactly like the original's
  single `model(X)` call; it is cached so predicting in batches doesn't change that. Runs therefore differ by that sample.
- **Bug fixes in the original:** `model = 'HBNN'` was overwritten by the model object inside the loop (a `TypeError` on the second
  config, so the original effectively completed only one config); `np.int` no longer exists in modern NumPy; Adam's `decay=` was removed
  in recent Keras (replaced by the equivalent `InverseTimeDecay` schedule).
- **Collapsed redundant loops:** with `multibivariate` data the `resources` (cpu/mem) and `clusters` loops all trained on identical data
  (they only changed the output file names), and `ts` was only used in the file name (the dataset hardcodes a 0.8 split). Here it is a
  single config and `ts` is a real split fraction; set `TRAIN_SPLITS = [0.8, 0.6, 0.4, 0.2]` to sweep.
- **Dropped:** `CUDA_VISIBLE_DEVICES`, unused imports (`talos`, `tensorflow_probability`, `plot_model`), the unused tuning branch,
  the weight-checkpoint saving, and the univariate/single-cluster paths.
- **Kaggle additions:** data read from the attached dataset, `/kaggle/working` output, compact logging, resumable runs.